In [1]:
import pandas as pd 
import numpy as np 
import glob
import os
from sap_download import download_stockout_prediction, download_inventory_overview
from datetime import datetime
from inventory_utils2 import filter_special_stock

In [4]:
stockout_path = "psi_input\품절예상조회\품절예상조회_04.06 17시12분.xlsx"
overview_path = "psi_input\재고개요\재고개요_04.06 17시13분.xlsx"
sf_path       = "psi_input\SF\SF_2603.xlsx"

df_stockout = pd.read_excel(stockout_path)
df_overview = pd.read_excel(overview_path)
df_sf = pd.read_excel(sf_path)
df_sf.columns = df_sf.columns.astype(str)


<>:3: SyntaxWarning: invalid escape sequence '\S'
<>:3: SyntaxWarning: invalid escape sequence '\S'
C:\Users\USER\AppData\Local\Temp\ipykernel_17996\220986695.py:3: SyntaxWarning: invalid escape sequence '\S'
  sf_path       = "psi_input\SF\SF_2603.xlsx"


In [6]:
# 품절예상조회 전처리 
df_stockout = df_stockout[["자재", "자재명", "3개월 평균출하", "당월출하"]].copy()
df_stockout.rename(columns = {"자재":"자재코드", "자재명":"자재내역", "3개월 평균출하":"3평판"}, inplace=True)
df_stockout["자재코드"] = df_stockout["자재코드"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
df_stockout["당월출하"] = pd.to_numeric(df_stockout["당월출하"], errors="coerce").fillna(0)
df_stockout["3평판"] = pd.to_numeric(df_stockout["3평판"], errors="coerce").fillna(0)

# SF 전처리 
today = datetime.today() 
year_month = today.strftime('%Y.%m')
df_sf = df_sf[["자재코드", "자재내역", "관리 채널", year_month]].copy()
df_sf.rename(columns = {"관리 채널":"관리채널", year_month:"SF"}, inplace=True)
df_sf["자재코드"] = df_sf["자재코드"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
df_sf["SF"] = pd.to_numeric(df_sf["SF"], errors="coerce").fillna(0)
df_sf = df_sf.groupby("자재코드").agg(자재내역=("자재내역", "first"), SF=("SF", "sum")).reset_index()
#df_sf["SF"] = df_sf["SF"].round().astype(int)

# 품절예상조회 + SF 병합
# df_standard의 자재내역 기본, 비어있는 경우만 df_sf 자재내역으로 채움
df_standard = pd.merge(df_stockout, df_sf, on="자재코드", how="outer", suffixes=("", "_sf"))
df_standard["자재내역"] = df_standard["자재내역"].fillna(df_standard["자재내역_sf"])
df_standard.drop(columns=["자재내역_sf"], inplace=True)

df_standard["3평판"]   = df_standard["3평판"].fillna(0)
df_standard["당월출하"] = df_standard["당월출하"].fillna(0)
df_standard["SF"]     = df_standard["SF"].fillna(0)

# 판매율 계산 (분모 0이면 NaN 처리)
df_standard["판매율(평판)"] = (df_standard["당월출하"] / df_standard["3평판"]).where(df_standard["3평판"] != 0)
df_standard["판매율(SF)"]  = (df_standard["당월출하"] / df_standard["SF"]).where(df_standard["SF"] != 0)

df_standard["3평판"] = df_standard["3평판"].astype(float)
df_standard["당월출하"] = df_standard["당월출하"].astype(float)
df_standard["SF"] = df_standard["SF"].astype(float)

# 판매율(SF) 높은 순 정렬
df_standard = df_standard.sort_values("판매율(SF)", ascending=False, na_position="last").reset_index(drop=True)
df_standard = df_standard[["자재코드", "자재내역", "3평판", "SF", "당월출하", "판매율(평판)", "판매율(SF)"]]

In [8]:
display(df_standard.head())

,자재코드,자재내역,3평판,SF,당월출하,판매율(평판),판매율(SF)
0,9302971,(단종)[임가공]RX_프레좀알엑스(30ml*3ea),709.667,5.887193,273.0,0.384687,46.371841
1,9311062,(단종)멜라B_매트커버팩트_23베이지_본품_19g,1909.667,17.115393,540.0,0.282772,31.550547
2,9310616,멜라_토닝원데이앰플_15ml,22504.667,100.000000,1868.0,0.083005,18.680000
3,9309798,DWEGF_코어부스팅아이크림_30ml(홈쇼핑),9998.000,331.546871,3547.0,0.354771,10.698337
4,9312895,[세트_임가공]멜라_토닝앰플쿠션_스페셜기획세트_21호(쿠팡기획세트),480.333,28.589707,216.0,0.449688,7.555167


In [9]:
#재고개요 전처리 
df_overview = df_overview[["자재","자재 내역", "저장 위치", "배치", "특별 재고", "기말 재고 수량"]].copy()
df_overview.rename(columns = {"자재":"자재코드", "자재 내역":"자재내역", "저장 위치":"저장위치", "배치":"배치", "특별 재고":"특별재고", "기말 재고 수량":"기말재고"}, inplace=True)
df_overview["자재코드"] = df_overview["자재코드"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
df_overview["기말재고"] = pd.to_numeric(df_overview["기말재고"], errors="coerce").fillna(0)

# 특별재고 제거
df_overview = filter_special_stock(df_overview)

# 자재코드- 저장위치로 grouping 
df_overview["자재코드"] = df_overview["자재코드"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
df_overview["저장위치"] = df_overview["저장위치"].fillna("알수없음")
df_overview["저장위치"] = df_overview["저장위치"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

df_overview = df_overview.groupby(["자재코드", "저장위치"], as_index = False).agg({
"자재내역" : "first",
"기말재고" : "sum"
})

In [10]:
display(df_overview.head())

,자재코드,저장위치,자재내역,기말재고
0,1000940,5000,[원료]DW-EGFLiposol(EGF-나노리포좀)250ppm 1g,117800.000
1,1000940,7000,[원료]DW-EGFLiposol(EGF-나노리포좀)250ppm 1g,0.000
2,1000940,알수없음,[원료]DW-EGFLiposol(EGF-나노리포좀)250ppm 1g,478219.014
3,1300271,알수없음,[원료]대웅제약_DW-EGF동결원액_4mg/1ml,1201.960
4,2301915,5000,(단종)[판촉물]쇼핑백_23년프로모션(245*95*230),13.000


In [11]:
df_overview.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296 entries, 0 to 1295
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   자재코드    1296 non-null   object 
 1   저장위치    1296 non-null   object 
 2   자재내역    1296 non-null   object 
 3   기말재고    1296 non-null   float64
dtypes: float64(1), object(3)
memory usage: 40.6+ KB


In [ ]:
df_major_WH = (
    df_overview[df_overview["저장위치"].isin(["5000", "6090"])]
    .groupby(["자재코드", "자재내역"])["기말재고"]
    .sum()
    .reset_index()
    .rename(columns={"기말재고": "천지+이지재고"})
)
